# Fraud Detection EDA: Certificate Verification Analysis

**Goal:** Explore fraud patterns in certificate verification logs and blockchain data

**Dataset:** 116 certificates with 517 verification attempts, 31 engineered features

**Key Questions:**
- What fraud patterns exist in verification data?
- Which features best distinguish fraud from legitimate certificates?
- Are there temporal or behavioral anomalies?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('✅ Libraries loaded')

## 1. Load & Explore Data

In [ ]:
# Load the exported CSV
# Note: Replace the filename with your actual CSV path
csv_path = '../data/ml/fraud_dataset_1773908139609.csv'  # Update this with your CSV name
df = pd.read_csv(csv_path)

print(f"✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Data info
print("Data Types & Missing Values:")
print(df.info())

In [ ]:
# Statistical summary
print("\nBasic Statistics:")
df.describe()

## 2. Fraud Distribution Analysis

In [ ]:
# Fraud label distribution
fraud_counts = df['likely_fraud_label'].value_counts()
fraud_pct = df['likely_fraud_label'].value_counts(normalize=True) * 100

print("Fraud Distribution:")
print(f"  Normal:  {fraud_counts.get(0, 0):3d} ({fraud_pct.get(0, 0):5.1f}%)")
print(f"  Fraud:   {fraud_counts.get(1, 0):3d} ({fraud_pct.get(1, 0):5.1f}%)")
print(f"  Total:   {len(df)}")
print(f"\n⚠️  Class Balance: {fraud_pct.get(1, 0):.1f}% fraud (imbalanced dataset)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
colors = ['#2ecc71', '#e74c3c']  # Green=Normal, Red=Fraud
fraud_counts.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Fraud Distribution (Count)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Label (0=Normal, 1=Fraud)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Normal', 'Fraud'], rotation=0)

# Pie chart
axes[1].pie(fraud_counts, labels=['Normal', 'Fraud'], autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Fraud Proportion (%)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Fraud distribution complete")

## 3. Feature Correlation with Fraud Label

In [ ]:
# Calculate correlations
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corr()['likely_fraud_label'].sort_values(ascending=False)

print("\nTop Features Correlated with Fraud:")
print(correlations[1:11])  # Skip fraud_label itself

# Top 15 correlations (positive and negative)
top_corr = pd.concat([
    correlations[correlations > 0][1:8],  # Top positive
    correlations[correlations < 0][:7]    # Top negative
])

plt.figure(figsize=(10, 6))
colors_corr = ['#e74c3c' if x > 0 else '#3498db' for x in top_corr.values]
top_corr.plot(kind='barh', color=colors_corr)
plt.title('Feature Correlation with Fraud Label', fontsize=12, fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\n✅ Correlation analysis complete")

## 4. Key Fraud Indicators Comparison

In [ ]:
# Compare key features between fraud and normal
key_features = [
    'suspicious_attempts',
    'max_suspicious_score',
    'failed_verifications',
    'unique_verifier_ips',
    'unique_user_agents',
    'total_verifications',
    'average_quiz_score'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, feature in enumerate(key_features):
    if feature in df.columns:
        normal = df[df['likely_fraud_label'] == 0][feature]
        fraud = df[df['likely_fraud_label'] == 1][feature]
        
        data_to_plot = [normal, fraud]
        axes[idx].boxplot(data_to_plot, labels=['Normal', 'Fraud'])
        axes[idx].set_title(feature, fontsize=10, fontweight='bold')
        axes[idx].set_ylabel('Value')
        axes[idx].grid(True, alpha=0.3)

# Hide extra subplots
for idx in range(len(key_features), len(axes)):
    axes[idx].axis('off')

plt.suptitle('Distribution: Fraud vs Normal', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n✅ Feature comparison complete")

## 5. Verification Behavior Analysis

In [ ]:
# Verification statistics by fraud status
print("\n" + "="*60)
print("VERIFICATION BEHAVIOR: Normal vs Fraud")
print("="*60)

comparison_features = [
    'total_verifications',
    'successful_verifications',
    'failed_verifications',
    'suspicious_attempts',
    'unique_verifier_ips',
    'unique_user_agents',
    'avg_response_time_ms',
    'max_suspicious_score'
]

for feature in comparison_features:
    if feature in df.columns:
        normal_mean = df[df['likely_fraud_label'] == 0][feature].mean()
        fraud_mean = df[df['likely_fraud_label'] == 1][feature].mean()
        diff_pct = ((fraud_mean - normal_mean) / normal_mean * 100) if normal_mean != 0 else 0
        
        print(f"\n{feature:30s}")
        print(f"  Normal: {normal_mean:10.2f}")
        print(f"  Fraud:  {fraud_mean:10.2f}  ({diff_pct:+.0f}%)")

print("\n" + "="*60)

## 6. Blockchain Integration Analysis

In [ ]:
# Blockchain features
print("\nBlockchain Integration Stats:")
print(f"Certificates with blockchain TX: {df['has_blockchain_tx'].sum()} / {len(df)}")
print(f"Avg blockchain delay (minutes): {df['blockchain_delay_minutes'].mean():.2f}")

# Blockchain vs Fraud
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Has blockchain TX
blockchain_fraud = pd.crosstab(
    df['has_blockchain_tx'],
    df['likely_fraud_label'],
    margins=False
)
blockchain_fraud.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Blockchain TX Status by Fraud Label', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Has Blockchain TX (0=No, 1=Yes)')
axes[0].set_ylabel('Count')
axes[0].legend(labels=['Normal', 'Fraud'])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Blockchain delay
normal_delay = df[df['likely_fraud_label'] == 0]['blockchain_delay_minutes']
fraud_delay = df[df['likely_fraud_label'] == 1]['blockchain_delay_minutes']
axes[1].hist([normal_delay, fraud_delay], label=['Normal', 'Fraud'], bins=20, color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Blockchain Delay Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Delay (minutes)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n✅ Blockchain analysis complete")

## 7. Completion & Performance Metrics

In [ ]:
# Course completion metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Average quiz score
df.boxplot(column='average_quiz_score', by='likely_fraud_label', ax=axes[0, 0])
axes[0, 0].set_title('Quiz Score Distribution')
axes[0, 0].set_xlabel('Fraud Status (0=Normal, 1=Fraud)')
axes[0, 0].set_ylabel('Average Score')
plt.sca(axes[0, 0])
plt.xticks([1, 2], ['Normal', 'Fraud'])

# Completion percentage
df.boxplot(column='completion_percentage', by='likely_fraud_label', ax=axes[0, 1])
axes[0, 1].set_title('Course Completion %')
axes[0, 1].set_xlabel('Fraud Status (0=Normal, 1=Fraud)')
axes[0, 1].set_ylabel('Completion %')
plt.sca(axes[0, 1])
plt.xticks([1, 2], ['Normal', 'Fraud'])

# Total time spent
df.boxplot(column='total_time_spent_min', by='likely_fraud_label', ax=axes[1, 0])
axes[1, 0].set_title('Study Time Distribution')
axes[1, 0].set_xlabel('Fraud Status (0=Normal, 1=Fraud)')
axes[1, 0].set_ylabel('Time (minutes)')
plt.sca(axes[1, 0])
plt.xticks([1, 2], ['Normal', 'Fraud'])

# Honors distribution
honors_counts = pd.crosstab(df['honors'], df['likely_fraud_label'])
honors_counts.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'])
axes[1, 1].set_title('Honors Distribution by Fraud')
axes[1, 1].set_xlabel('Honors (0=No, 1=Yes)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].legend(labels=['Normal', 'Fraud'])
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("\n✅ Performance metrics analysis complete")

## 8. Fraud Patterns Summary

In [ ]:
print("\n" + "="*70)
print("KEY FRAUD PATTERNS DISCOVERED")
print("="*70)

# Pattern 1: Suspicious IP behavior
fraud_df = df[df['likely_fraud_label'] == 1]
normal_df = df[df['likely_fraud_label'] == 0]

print("\n🔴 Pattern 1: Suspicious IP/User Agent Behavior")
print(f"  Fraud avg unique IPs: {fraud_df['unique_verifier_ips'].mean():.1f}")
print(f"  Normal avg unique IPs: {normal_df['unique_verifier_ips'].mean():.1f}")
print(f"  →  Fraud shows {(fraud_df['unique_verifier_ips'].mean() / normal_df['unique_verifier_ips'].mean()):.1f}x more unique IPs")

print("\n🔴 Pattern 2: High Suspicious Scores")
print(f"  Fraud max suspicious score: {fraud_df['max_suspicious_score'].mean():.1f}")
print(f"  Normal max suspicious score: {normal_df['max_suspicious_score'].mean():.1f}")

print("\n🔴 Pattern 3: Failed Verifications")
print(f"  Fraud avg failed verifications: {fraud_df['failed_verifications'].mean():.1f}")
print(f"  Normal avg failed verifications: {normal_df['failed_verifications'].mean():.1f}")

print("\n🔴 Pattern 4: Suspicious Attempts")
print(f"  Fraud avg suspicious attempts: {fraud_df['suspicious_attempts'].mean():.1f}")
print(f"  Normal avg suspicious attempts: {normal_df['suspicious_attempts'].mean():.1f}")

print("\n🟢 Pattern 5: Certificate Quality")
print(f"  Fraud avg quiz score: {fraud_df['average_quiz_score'].mean():.1f}")
print(f"  Normal avg quiz score: {normal_df['average_quiz_score'].mean():.1f}")
print(f"  →  Fraud shows lower academic performance")

print("\n" + "="*70)
print("\n✅ EDA complete! Ready for XGBoost modeling.")

## 9. Data Preparation for Modeling

In [ ]:
# Save processed data for modeling
df.to_csv('../data/ml/fraud_dataset_processed.csv', index=False)
print("✅ Processed dataset saved for XGBoost training")
print(f"   Shape: {df.shape}")
print(f"   Location: ../data/ml/fraud_dataset_processed.csv")
print("\n📌 Next: Open '01_xgboost_training.py' for model training")